In [1]:
def run_pipeline(image_path):

    # ========================================================================
    # БЛОК 1. ИМПОРТ БИБЛИОТЕК
    # ======================================================================== 

    import cv2
    import os
    import numpy as np
    import matplotlib.pyplot as plt
    from skimage.morphology import skeletonize

    # ========================================================================
    # БЛОК 2. ПРЕДОБРАБОТКА ИЗОБРАЖЕНИЯ
    # ======================================================================== 

    # 1. Загрузка анализируемого изображения в градациях серого
    img = cv2.imread(image_path, 0)

    # Извлечение имени файла (без пути и расширения)
    image_name = os.path.basename(image_path)
    image_name = os.path.splitext(image_name)[0]

    # Формирование директории для сохранения результатов обработки
    processed_dir = os.path.join("..", "Data", "Processed", image_name)
    os.makedirs(processed_dir, exist_ok=True)

    # Вспомогательные функции визуализации и сохранения
    def show(img, title="Image", cmap='gray',
            save=False, save_dir=None, filename=None,
            display=False):
        """
        Универсальная функция визуализации промежуточных результатов.

        Параметры:
        - img: изображение
        - title: заголовок
        - cmap: цветовая карта (по умолчанию grayscale)
        - save: сохранять ли изображение
        - display: отображать ли в ноутбуке
        """

        plt.figure(figsize=(15, 15))
        plt.imshow(img, cmap=cmap)
        plt.title(title, fontsize=28)
        plt.axis('off')
        
        if save and save_dir and filename:
            save_path = os.path.join(save_dir, filename)
            plt.savefig(save_path, bbox_inches='tight', dpi=300)
        
        if display:
            plt.show()
        
        plt.close()

    def show_graph(img, title="Image", cmap='gray',
            save=False, save_dir=None, filename=None,
            display=False):
        """
        Визуализация графовых представлений (узлы + рёбра).
        Используется отдельная функция для управления параметрами отображения графа.
        """

        plt.figure(figsize=(15, 15))
        plt.imshow(img, cmap=cmap)
        plt.title(title, fontsize=32)
        plt.axis('off')
        
        if save and save_dir and filename:
            save_path = os.path.join(save_dir, filename)
            plt.savefig(save_path, bbox_inches='tight', dpi=300)
        
        if display:
            plt.show()
        
        plt.close()

    # 1. Проверка корректности загрузки и сохранение исходного изображения
    if img is None:
        print("Ошибка: Изображение не найдено. Загрузите файл в папку и укажите путь.")
    else:
        show(
        img,
        "1. Исходное изображение",
        save=True,
        save_dir=processed_dir,
        filename="01_original_img.png"
        )

        # 2. Адаптивная бинаризация (устойчива к неравномерному освещению):
        binary = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                    cv2.THRESH_BINARY_INV, 11, 2)
        show(
        binary,
        "2. Адаптивная бинаризация",
        save=True,
        save_dir=processed_dir,
        filename="02_binary.png"
        )

        # 3. Очистка шумов на изображении
        # 3.1 Очистка шумов на внешних контурах
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            if cv2.contourArea(cnt) < 100:
                cv2.drawContours(binary, [cnt], -1, 0, -1)
        # 3.2 Очистка шумов на внутренних контурах
        contours, _ = cv2.findContours(binary, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            if cv2.contourArea(cnt) < 10:
                cv2.drawContours(binary, [cnt], -1, 0, -1)

        # 4. Скелетизация бинарного изображения
        skeleton = skeletonize(binary // 255)
        skeleton_visual = (skeleton * 255).astype(np.uint8)
        show(
        skeleton_visual,
        "3. Результат скелетизации",
        save=True,
        save_dir=processed_dir,
        filename="03_skeleton.png"
        )

    # ========================================================================
    # БЛОК 3. ГЕОМЕТРИЧЕСКИЙ АНАЛИЗ (ВЫЧИСЛЕНИЕ УГЛОВ)
    # ======================================================================== 
    
    import math
    import numpy as np

    def calculate_angle(node_center, node_A, node_B, graph):
        """
        Вычисление угла между двумя рёбрами графа, исходящими из одного узла.

        Параметры:
        - node_center: ID центрального узла (шарнира)
        - node_A, node_B: ID соседних узлов, формирующих два ребра
        - graph: объект графа NetworkX (построенный через sknw)

        Возвращает:
        - угол между рёбрами в градусах (float)
        """
        # 1. Извлечение координат узлов
        y_c, x_c = graph.nodes[node_center]['o']
        y_a, x_a = graph.nodes[node_A]['o']
        y_b, x_b = graph.nodes[node_B]['o']
        
        # 2. Формирование векторов, направленных от центрального узла к соседним
        v1 = np.array([x_a - x_c, y_a - y_c])
        v2 = np.array([x_b - x_c, y_b - y_c])
        
        # 3. Вычисление норм (длин векторов)
        norm_v1 = np.linalg.norm(v1)
        norm_v2 = np.linalg.norm(v2)
        
        # Защита от совпадающих точек
        if norm_v1 == 0 or norm_v2 == 0:
            return 0.0
        
        # 4. Вычисление угла через скалярное произведение векторов
        cos_theta = np.dot(v1, v2) / (norm_v1 * norm_v2)
        cos_theta = np.clip(cos_theta, -1.0, 1.0)

        angle_rad = math.acos(cos_theta)
        angle_deg = math.degrees(angle_rad)
        
        return round(angle_deg, 1)
    
    # ========================================================================
    # БЛОК 4. ПОСТРОЕНИЕ И ФИЛЬТРАЦИЯ ТОПОЛОГИЧЕСКОГО ГРАФА
    # ========================================================================

    import sknw
    import networkx as nx
    import numpy as np
    import matplotlib.pyplot as plt
    import cv2
    # Приведение скелета к бинарному виду
    skeleton_binary = (skeleton_visual > 0).astype(np.uint8)

    # Построение графа:
    # узлы - точки пересечений и окончаний линий
    # ребра - линии между узлами
    graph = sknw.build_sknw(skeleton_binary, multi=True)

    # -----------------------------------------------
    # Фильтрация графа (Удаление короткого шума)
    # -----------------------------------------------

    img_diagonal = np.sqrt(img.shape[0]**2 + img.shape[1]**2)

    # Порог длины для коротких элементов
    min_edge_length = img_diagonal * 0.025  # 2.5% от диагонали

    edges_to_remove = []
    for u, v, key, data in graph.edges(keys=True, data=True):
        if data['weight'] < min_edge_length:
            edges_to_remove.append((u, v))

    # Удаляем короткие ребра
    graph.remove_edges_from(edges_to_remove)

    # Удаляем узлы, оставщиеся без ребер
    isolated_nodes = list(nx.isolates(graph))
    graph.remove_nodes_from(isolated_nodes)

    # -----------------------------------------------
    # Фильтрация малых компонент графа
    # -----------------------------------------------

    min_nodes = 3  # минимальный размер компоненты

    nodes_to_keep = set()

    for comp in nx.connected_components(graph):
        subgraph = graph.subgraph(comp)

        # Приводим к обычному графу для анализа циклов
        simple_subgraph = nx.Graph(subgraph)
        has_cycle = len(list(nx.cycle_basis(simple_subgraph))) > 0
        if len(comp) >= min_nodes or has_cycle:
            nodes_to_keep.update(comp)
   
    # Формирование очищенного графа 
    graph = graph.subgraph(nodes_to_keep).copy()

    # -----------------------------------------------
    # Восстановление разорванных линий
    # -----------------------------------------------

    # Порог расстояния между концами линий
    threshold_dist = max(11, int(img_diagonal * 0.005))

    # Поиск конечных узлов (степень = 1)
    end_nodes = [n for n in graph.nodes() if graph.degree(n) == 1]

    # Объединение близких концов
    for i in range(len(end_nodes)):
        for j in range(i + 1, len(end_nodes)):

            n1 = end_nodes[i]
            n2 = end_nodes[j]

            y1, x1 = graph.nodes[n1]['o']
            y2, x2 = graph.nodes[n2]['o']
            x1, y1 = float(x1), float(y1)
            x2, y2 = float(x2), float(y2)


            dist = np.sqrt((x1 - x2)**2 + (y1 - y2)**2)

            if dist < threshold_dist:

                pts = [
                    [float(y1), float(x1)],
                    [float(y2), float(x2)]
                ]

                graph.add_edge(
                    n1,
                    n2,
                    weight=float(dist),
                    artificial=True,
                    pts=pts
                )

    # -----------------------------------------------
    # Удаление висячих узлов
    # -----------------------------------------------

    # Удаляем узлы степени 1, если рядом нет другого конца, значит это шум
    nodes_to_remove = []

    for node in graph.nodes():
        if graph.degree(node) == 1:
            y1, x1 = graph.nodes[node]['o']

            has_neighbor_close = False

            for other in graph.nodes():
                if other == node:
                    continue

                if graph.degree(other) == 1:
                    y2, x2 = graph.nodes[other]['o']
                    x2, y2 = float(x2), float(y2)
                    
                    dist = np.sqrt((x1 - x2)**2 + (y1 - y2)**2)

                    if dist < threshold_dist:
                        has_neighbor_close = True
                        break

            if not has_neighbor_close:
                nodes_to_remove.append(node)

    graph.remove_nodes_from(nodes_to_remove)

    # -----------------------------------------------
    # Итеративное удаление всех висячих элементов
    # -----------------------------------------------

    while True:
        nodes_to_remove = [n for n in graph.nodes() if graph.degree(n) == 1]
        
        if not nodes_to_remove:
            break  # больше нет висячих узлов
        
        graph.remove_nodes_from(nodes_to_remove)

    # Финальная очистка изолированных узлов
    graph.remove_nodes_from(list(nx.isolates(graph)))

    # -----------------------------------------------
    # Визуализация графа поверх изображения
    # -----------------------------------------------

    output_img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

    # Ребра
    for s, e, key, data in graph.edges(keys=True, data=True):
        ps = data['pts']

        for i in range(len(ps) - 1):
            pt1 = (int(ps[i][1]), int(ps[i][0]))
            pt2 = (int(ps[i+1][1]), int(ps[i+1][0]))
            cv2.line(output_img, pt1, pt2, (0, 0, 255), 1)

    # Узлы
    for node in graph.nodes():
        y, x = graph.nodes[node]['o']
        cv2.circle(output_img, (int(x), int(y)), 1, (255, 0, 0), -1)

    # Сохранение результата
    show(
        cv2.cvtColor(output_img, cv2.COLOR_BGR2RGB),
        "Топологический граф поверх чертежа",
        cmap=None,
        save=True,
        save_dir=processed_dir,
        filename="04_graph_overlay.png"
        )

    # ========================================================================
    # БЛОК 5. МЕТРИЧЕСКИЙ АНАЛИЗ ТОПОЛОГИЧЕСКОГО ГРАФА
    # ========================================================================

    from itertools import combinations
    import numpy as np
    import math

    metrics_data = []

    # -----------------------------------------------
    # Вспомогательная функция получения геометрии ребра
    # -----------------------------------------------
    def get_pts(graph, u, v, key):
        data = graph[u][v][key]
        pts = data.get("pts", None)

        if pts is None or len(pts) < 2:
            return None
        return np.array(pts)

    # -----------------------------------------------
    # Вспомогательная функция получения направления ребра
    # -----------------------------------------------
    def get_edge_vector(graph, u, v, key, center_node):
        pts = get_pts(graph, u, v, key)
        if pts is None:
            return None

        y_c, x_c = graph.nodes[center_node]['o']
        y1, x1 = pts[0]
        y2, x2 = pts[-1]

        if np.linalg.norm([x1 - x_c, y1 - y_c]) < np.linalg.norm([x2 - x_c, y2 - y_c]):
            vec = np.array([x2 - x1, y2 - y1])
        else:
            vec = np.array([x1 - x2, y1 - y2])

        norm = np.linalg.norm(vec)
        if norm == 0:
            return None
        return vec / norm

    # -----------------------------------------------
    # Основной цикл анализа углов графа
    # -----------------------------------------------
    for node in graph.nodes():

        edges = list(graph.edges(node, keys=True, data=True))

        # Анализ проводится только для узлов с >= 2 ребрами
        if len(edges) < 2:
            continue

        # Перебор пар рёбер
        for e1, e2 in combinations(edges, 2):

            u1, v1, k1, d1 = e1
            u2, v2, k2, d2 = e2
            n1 = v1 if u1 == node else u1
            n2 = v2 if u2 == node else u2
            len_1 = d1.get('weight', 0)
            len_2 = d2.get('weight', 0)

            if len_1 == 0 or len_2 == 0:
                continue
            ratio = len_1 / len_2

            # -----------------------------------------------
            # Вычисление угла через векторы
            # -----------------------------------------------

            vec1 = get_edge_vector(graph, node, n1, k1, node)
            vec2 = get_edge_vector(graph, node, n2, k2, node)

            if vec1 is None or vec2 is None:
                continue
            cos_theta = np.clip(np.dot(vec1, vec2), -1.0, 1.0)
            angle = math.degrees(math.acos(cos_theta))

            # -----------------------------------------------
            # Сохранение метрик
            # -----------------------------------------------

            metrics_data.append({
                "joint_node": node,
                "connected_parts": [n1, n2],
                "lengths_px": [round(len_1, 1), round(len_2, 1)],
                "length_ratio": round(ratio, 3),
                "angle_degrees": round(angle, 2)
            })

    # ========================================================================
    # БЛОК 6. ФОРМИРОВАНИЕ СТРУКТУРИРОВАННОГО ПРЕДСТАВЛЕНИЯ (JSON)
    # ========================================================================

    import json
    import os

    # Создание структуры данных для агента
    agent_payload = {
        "metadata": {
            "img_name": image_name,
            "description": "Структурно-параметрическая модель 2D-чертежа",
            "task": "Анализ геометрии и пропорций на 2D-чертеже"
        },
        "topology": {
            "nodes": [],
            "edges": []
        },
        "kinematics_and_metrics": metrics_data
    }

    # Формирование узлов графа
    for node_id in graph.nodes():
        y, x = graph.nodes[node_id]['o']
        # Классификация узла: если 1 связь - это конец детали, если >1 - соединение
        node_type = "end_point" if graph.degree(node_id) == 1 else "joint"
        
        agent_payload["topology"]["nodes"].append({
            "id": int(node_id),
            "type": node_type,
            "coordinates_xy": [int(x), int(y)]
        })

    # Формирование ребер
    for u, v, data in graph.edges(data=True):
        length = data['weight']
        agent_payload["topology"]["edges"].append({
            "source_node": int(u),
            "target_node": int(v),
            "length_pixels": round(length, 1)
        })

    # ========================================================================
    # БЛОК 7 СЕМАНТИЧЕСКИЙ СЛОЙ
    # ========================================================================

    # -----------------------------------------------
    # Функция классификации угла
    # -----------------------------------------------

    def classify_angle(angle):
        if abs(angle - 90) < 10:
            return "Прямой"
        elif abs(angle - 180) < 10:
            return "Развернутый"
        elif angle < 80:
            return "Острый"
        else:
            return "Тупой"
        
    # -----------------------------------------------
    # Определение структуры онтологии
    # -----------------------------------------------

    ontology = {
        "classes": {
            "Node": {},
            "Joint": {"subclass_of": "Node"},
            "EndPoint": {"subclass_of": "Node"},
            "LineSegment": {},
            "Angle": {},
            "LengthRatio": {}
        },
        "entities": {
            "nodes": [],
            "line_segments": [],
            "angles": [],
            "ratios": []
        }
    }

    # -----------------------------------------------
    # Узлы → онтология
    # -----------------------------------------------

    for node_id in graph.nodes():
        y, x = graph.nodes[node_id]['o']
        degree = graph.degree(node_id)

        node_type = "EndPoint" if degree == 1 else "Joint"

        ontology["entities"]["nodes"].append({
            "id": f"Node_{node_id}",
            "type": node_type,
            "coordinates": [int(x), int(y)]
        })


    # -----------------------------------------------
    # Ребра → LineSegmet
    # -----------------------------------------------

    edge_id_map = {}
    edge_counter = 0

    for u, v, data in graph.edges(data=True):
        edge_id = f"Line_{edge_counter}"
        edge_id_map[(u, v)] = edge_id
        edge_id_map[(v, u)] = edge_id

        ontology["entities"]["line_segments"].append({
            "id": edge_id,
            "type": "LineSegment",
            "connects": [f"Node_{u}", f"Node_{v}"],
            "length": round(data['weight'], 1)
        })

        edge_counter += 1


    # -----------------------------------------------
    # Углы и пропорции
    # -----------------------------------------------

    angle_counter = 0
    ratio_counter = 0

    for m in metrics_data:
        node = m["joint_node"]
        n1, n2 = m["connected_parts"]

        line_1 = edge_id_map.get((node, n1))
        line_2 = edge_id_map.get((node, n2))

        if line_1 is None or line_2 is None:
            continue

        angle = m["angle_degrees"]
        ratio = m["length_ratio"]

        # Угол
        ontology["entities"]["angles"].append({
            "id": f"Angle_{angle_counter}",
            "type": "Angle",
            "at_node": f"Node_{node}",
            "between": [line_1, line_2],
            "value_degrees": angle,
            "semantic_type": classify_angle(angle)
        })

        angle_counter += 1

        # Пропорция
        ontology["entities"]["ratios"].append({
            "id": f"Ratio_{ratio_counter}",
            "type": "LengthRatio",
            "at_node": f"Node_{node}",
            "between": [line_1, line_2],
            "value": ratio
        })

        ratio_counter += 1

    # Интеграция в общий JSON
    agent_payload["ontology"] = ontology

    # Сохранение
    json_filename = f"{image_name}_agent_data.json"
    output_json_path = os.path.join("..", "Output", json_filename)
    with open(output_json_path, 'w', encoding='utf-8') as json_file:
        json.dump(agent_payload, json_file, ensure_ascii=False, indent=4)

    # ========================================================================
    # БЛОК 8. ВИЗУАЛИЗАЦИЯ ТОПОЛОГИИ
    # ========================================================================

    import cv2
    import numpy as np

    # Создание пустого изображения
    output_img = np.ones((img.shape[0], img.shape[1], 3), dtype=np.uint8) * 255

    # -----------------------------------------------
    # Отрисовка элементов
    # -----------------------------------------------

    # Отрисовка ребер
    for edge in graph.edges(data=True):
        u, v, data = edge
        pts = data.get('pts', None)
        if pts is None:
            continue
        pts = np.asarray(pts)
        for i in range(len(pts) - 1):
            y1, x1 = pts[i]
            y2, x2 = pts[i + 1]
            pt1 = (int(x1), int(y1))
            pt2 = (int(x2), int(y2))
            cv2.line(output_img, pt1, pt2, (0, 0, 255), 1)

    # Отрисовка узлов
    for node in graph.nodes():
        y, x = graph.nodes[node]['o']
        degree = graph.degree(node)
        color = (0, 255, 0) if degree == 1 else (255, 0, 0) # Зеленый - конец линии, синий - соединение
        cv2.circle(output_img, (int(x), int(y)), 2, color, -1)

    # -----------------------------------------------
    # Сохранение
    # -----------------------------------------------

    show(
        cv2.cvtColor(output_img, cv2.COLOR_BGR2RGB),
        "Топологический граф исходного чертежа",
        cmap=None,
        save=True,
        save_dir=processed_dir,
        filename="05_graph_only.png"
        )
    
    # ========================================================================
    # БЛОК 9. АБСТРАКТНАЯ ВИЗУАЛИЗАЦИЯ ГРАФА
    # ========================================================================

    import networkx as nx
    import matplotlib.pyplot as plt

    G_vis = nx.Graph()

    # Узлы
    for node_id in graph.nodes():
        degree = graph.degree(node_id)
        if degree == 1:
            node_type = "end"
        else:
            node_type = "joint"
        G_vis.add_node(f"N{node_id}", type=node_type)

    # Ребра
    for u, v in graph.edges():
        G_vis.add_edge(f"N{u}", f"N{v}")

    # Абстрактное расположение
    pos = nx.spring_layout(G_vis, k=1, seed = 1)

    # Цвета узлов
    node_colors = []
    for n in G_vis.nodes(data=True):
        if n[1]["type"] == "end":
            node_colors.append("green")
        else:
            node_colors.append("red")

    # Отрисовка графа
    import matplotlib.pyplot as plt
    import numpy as np

    def draw_graph_to_image(G, pos, node_colors):
        fig = plt.figure(figsize=(8, 8))
        nx.draw(
            G,
            pos,
            with_labels=True,
            node_color=node_colors,
            node_size=400,
            edge_color="black",
            width=1.5
        )
        plt.title("Абстрактное представление графа", fontsize=64)
        fig.canvas.draw()
        img = np.asarray(fig.canvas.renderer.buffer_rgba())
        img = img[:, :, :3]
        plt.close(fig)
        return img
    graph_img = draw_graph_to_image(G_vis, pos, node_colors)

    # Сохранение
    show_graph(
        graph_img,
        "Абстрактное представление графа",
        cmap=None,
        save=True,
        save_dir=processed_dir,
        filename="06_abstract_graph.png",
        display=False
    )



In [3]:
from tqdm import tqdm
import time
import os

# ========================================================================
# БЛОК 10. ПАКЕТНАЯ ОБРАБОТКА ИЗОБРАЖЕНИЙ
# ========================================================================

# Путь к директории с входными изображениями
input_dir = "../Data/Raw"

files = sorted([
    f for f in os.listdir(input_dir)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
])

# Проверка наличия данных
if not files:
    print("Ошибка. В директории отсуствуют фотографии для анализа.")
else:
    # Основной цикл
    for filename in tqdm(files, desc="Pipeline execution"):
        image_path = os.path.join(input_dir, filename)

        start = time.perf_counter()
        run_pipeline(image_path)
        elapsed = time.perf_counter() - start

        print(f"[OK] {filename} | processing time: {elapsed:.2f}s")
print("Готво. Все файлы успешно проанализированы.")


Pipeline execution:  17%|█▋        | 1/6 [00:12<01:01, 12.36s/it]

[OK] 2D-draw.jpg | processing time: 12.36s


Pipeline execution:  33%|███▎      | 2/6 [00:17<00:32,  8.08s/it]

[OK] 2D-draw_1.png | processing time: 5.08s


Pipeline execution:  50%|█████     | 3/6 [00:24<00:22,  7.53s/it]

...

[OK] rus_164.png | processing time: 6.85s


Pipeline execution:  83%|████████▎ | 5/6 [00:38<00:07,  7.15s/it]

[OK] rus_438.png | processing time: 6.94s


Pipeline execution: 100%|██████████| 6/6 [00:45<00:00,  7.59s/it]

[OK] rus_439.png | processing time: 7.42s
Готво. Все файлы успешно проанализированы.
